# 🎞️ ROTBOTS — Assemble
## Videos + Audio + Music + Subtitles + Credits → Final Video

---

All features can be toggled on/off. The pipeline works with or without music, subtitles, and credits.

> **🤔** How does editing, music, and subtitles change the meaning of the same footage?

---

In [ ]:
# SETUP
import json, subprocess, shutil, os
from pathlib import Path
from IPython.display import display, Markdown, Video
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)
print('✅ Setup complete')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'session_info.json').exists()])
if not sessions: print('❌ No sessions!')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
VIDEOS_DIR = SESSION_DIR / 'videos'
AUDIO_DIR = SESSION_DIR / 'audio'
sub_file = SESSION_DIR / 'subtitles.ass'
video_files = sorted(VIDEOS_DIR.glob('scene_*.mp4')) if VIDEOS_DIR.exists() else []
audio_files = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []
essay = None
if (SESSION_DIR/'essay_script.json').exists():
    with open(SESSION_DIR/'essay_script.json') as f: essay = json.load(f)
print(f'✅ Session: {SESSION_NAME}')
print(f'   🎬 {len(video_files)} videos | 🎙️ {len(audio_files)} narrations | 📝 Subtitles: {"✅" if sub_file.exists() else "—"}')
if not video_files: print('❌ No videos!')

In [ ]:
# ============================================================
# ASSEMBLY OPTIONS — Toggle features on/off
# ============================================================

ENABLE_NARRATION = True      # Overlay narration audio
ENABLE_MUSIC = False         # Add background music
ENABLE_SUBTITLES = False     # Burn TikTok-style subtitles (needs 07_Subtitles)
ENABLE_CREDITS = True        # Rolling credits at the end

# Music settings (only used if ENABLE_MUSIC = True)
MUSIC_SOURCE = 'generate'    # 'generate' = AI via fal.ai | 'upload' = your own MP3
MUSIC_PROMPT = 'Ambient cinematic background music, slow contemplative mood'
MUSIC_VOLUME = 0.10          # 0.0–1.0 (0.10 = 10% volume under narration)

print('🎬 Assembly options:')
print(f'   Narration:  {"✅" if ENABLE_NARRATION else "❌"}')
print(f'   Music:      {"✅ " + MUSIC_SOURCE if ENABLE_MUSIC else "❌"}')
print(f'   Subtitles:  {"✅" if ENABLE_SUBTITLES else "❌"}')
print(f'   Credits:    {"✅" if ENABLE_CREDITS else "❌"}')

In [ ]:
# HELPERS
def dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0
def ff(cmd,desc=''):
    if desc: print(f'   {desc}...',end=' ',flush=True)
    r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
    if r.returncode==0:
        if desc: print('✅')
        return True
    print(f'❌ {r.stderr[:300]}'); return False

In [ ]:
# STEP 1: NORMALIZE
print('🔧 Step 1: Normalizing...')
norm = []
for v in video_files:
    out = TEMP / v.name
    if ff(['ffmpeg','-y','-i',str(v),'-vf','scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black','-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an',str(out)], v.name):
        norm.append(out)
print(f'✅ {len(norm)} normalized')

In [ ]:
# STEP 2: MUSIC (if enabled)
music_path = None
if ENABLE_MUSIC:
    if MUSIC_SOURCE == 'generate':
        print('🎵 Step 2: Generating music via fal.ai...')
        try:
            import fal_client, time, requests
            if not os.environ.get('FAL_KEY'):
                FAL_KEY = ''  # <-- Paste fal.ai key if not set in 05_Generate
                if FAL_KEY: os.environ['FAL_KEY'] = FAL_KEY
                else: raise Exception('Set FAL_KEY or run 05_Generate first')
            total_dur = sum(dur(v) for v in norm) + 10
            print(f'   Prompt: {MUSIC_PROMPT[:60]}...')
            print(f'   Duration: {int(total_dur)}s')
            t0 = time.time()
            result = fal_client.subscribe('beatoven/music-generation', arguments={'prompt':MUSIC_PROMPT,'duration':min(150,int(total_dur))})
            url = None
            if isinstance(result,dict):
                url = result.get('audio',{}).get('url') if isinstance(result.get('audio'),dict) else result.get('audio') or result.get('url')
            if url:
                music_path = TEMP / 'music.wav'
                with open(music_path,'wb') as f: f.write(requests.get(url,timeout=120).content)
                print(f'   ✅ Generated ({time.time()-t0:.0f}s)')
            else: print('   ⚠️ No audio URL')
        except Exception as e: print(f'   ❌ {e}'); print('   Continuing without music.')
    elif MUSIC_SOURCE == 'upload':
        print('🎵 Step 2: Upload music...')
        from google.colab import files
        for fn, content in files.upload().items():
            music_path = TEMP / 'music.mp3'
            with open(music_path,'wb') as f: f.write(content)
            print(f'   ✅ {fn} ({len(content)//1024}KB)'); break
else: print('🎵 Step 2: Music — skipped')

In [ ]:
# STEP 3: CREDITS (if enabled)
credits_path = None
if ENABLE_CREDITS:
    print('📜 Step 3: Generating credits...')
    title = essay.get('title','Untitled') if essay else 'Untitled'
    sources = essay.get('credits',{}).get('sources',[]) if essay else []
    lines = [title,'','Generated by ROTBOTS','Content Machine Workshop','','— Sources —']
    for s in sources: lines.append(s[:60]+'...' if len(s)>60 else s)
    lines += ['','— Pipeline —','LLM: Groq','Video: fal.ai','Voice: Edge-TTS','Composition: FFmpeg','','LI-MA TDA 2026, Amsterdam']
    D=8; LH=42; spd=(720+len(lines)*LH)/D
    flt = [f"drawtext=text='{l.replace(chr(39),chr(8217)).replace(chr(58),chr(92)+chr(58))}':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{spd:.1f}*t" for i,l in enumerate(lines) if l]
    credits_path = TEMP / 'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={D}:r=24','-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)],'Credits')
else: print('📜 Step 3: Credits — skipped')

In [ ]:
# STEP 4: CONCATENATE
print('🎬 Step 4: Concatenating...')
cf = TEMP / 'concat.txt'
with open(cf,'w') as f:
    for v in norm: f.write(f"file '{v}'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")
concat_out = TEMP / 'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)],'Concat')
print(f'   Duration: {dur(concat_out):.1f}s')

In [ ]:
# STEP 5: AUDIO MIX
audio_out = TEMP / 'with_audio.mp4'
has_narration = ENABLE_NARRATION and audio_files
has_music = music_path and music_path.exists()

if has_narration:
    print('🎙️ Step 5: Mixing audio...')
    acf = TEMP / 'audio_concat.txt'
    with open(acf,'w') as f:
        for a in audio_files: f.write(f"file '{a}'\n")
    narr = TEMP / 'narration.mp3'
    ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(narr)],'Combining narration')
    if has_music:
        mixed = TEMP / 'mixed.mp3'
        ff(['ffmpeg','-y','-i',str(narr),'-i',str(music_path),'-filter_complex',
            f'[0:a]volume=1.0[n];[1:a]volume={MUSIC_VOLUME}[m];[n][m]amix=inputs=2:duration=longest[out]',
            '-map','[out]','-c:a','libmp3lame',str(mixed)],'Mixing narration + music')
        ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(mixed),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Adding to video')
    else:
        ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(narr),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Adding narration')
elif has_music:
    print('🎵 Step 5: Music only...')
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(music_path),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Adding music')
else:
    print('ℹ️ Step 5: No audio')
    shutil.copy2(concat_out, audio_out)

In [ ]:
# STEP 6: BURN SUBTITLES (if enabled)
final = SESSION_DIR / 'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    print('📝 Step 6: Burning subtitles...')
    local_ass = TEMP / 'subtitles.ass'
    shutil.copy2(sub_file, local_ass)
    ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'ass={local_ass}',
        '-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)],'Burning subtitles')
elif ENABLE_SUBTITLES:
    print('📝 Step 6: Subtitles enabled but no subtitles.ass found — run 07_Subtitles first')
    shutil.copy2(audio_out, final)
else:
    print('📝 Step 6: Subtitles — skipped')
    shutil.copy2(audio_out, final)

if final.exists():
    d=dur(final); sz=final.stat().st_size/(1024*1024)
    print(f'\n✅ Final video: {d:.1f}s, {sz:.1f}MB')
    print(f'   📁 {final}')

---
## 🎬 Watch!

In [ ]:
title = essay.get('title','Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown(f'# 🎬 {title}\n*Generated by ROTBOTS*\n\n---'))
    try: display(Video(str(final),embed=True,width=720))
    except: print(f'Download: My Drive/rotbots_workshop/{SESSION_NAME}/final_video.mp4')
else: print('❌ No video')

In [ ]:
# DOWNLOAD
DOWNLOAD = False
if DOWNLOAD and final.exists():
    from google.colab import files; files.download(str(final))
else:
    print(f'📁 Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')
    print(f'   Set DOWNLOAD = True above.')

---
## 📊 Summary

In [ ]:
print('📊 Pipeline Summary')
print('='*50)
st=[]
if (SESSION_DIR/'summaries.json').exists():
    with open(SESSION_DIR/'summaries.json') as f: d=json.load(f)
    st.append(f'📥 {len(d.get("sources",[]))} sources')
if essay:
    tw=sum(len(s.get('narration','').split()) for c in essay.get('chapters',[]) for s in c.get('sections',[]))
    st.append(f'✍️ "{essay.get("title","")}" ({tw}w)')
st.append(f'🎬 {len(video_files)} clips')
st.append(f'🎙️ {len(audio_files)} narrations')
if has_music: st.append('🎵 Background music')
if ENABLE_SUBTITLES and sub_file.exists(): st.append('📝 TikTok subtitles')
if ENABLE_CREDITS: st.append(f'📜 Credits ({len(sources)} sources)')
if final.exists(): st.append(f'🎞️ Final: {dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')
for s in st: print(f'   {s}')
print(f'\n🤖 All from one topic.')

---
## 🏁 That's it!

The pipeline: sources → essay → storyboard → prompts → voice → video → music → subtitles → credits → final.

**Every step: automated decisions.** The question isn't whether AI can make content. It's: **what does that mean?**

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026, Amsterdam*